In [0]:
%run "./00_config"

## 01. Core Functions


## Metadata
- Reads `Is_Active = 1` rows from the control table (optionally filtered by
  `DataMart`), builds the full source path, and excludes any file already logged as `success`
  in the log table for the same target (unless `force_reload = True`)

In [0]:
def get_pending_files():
    control_df = spark.table(control_table).filter(col("Is_Active") == 1)

    if datamart_filter is not None:
        control_df = control_df.filter(col("DataMart") == datamart_filter)

    control_df = control_df.dropDuplicates(
        ["Target_Catalog", "Target_Schema", "Target_Table_Name", "Source_Path", "Source_File_Name"]
    )

    if force_reload:
        return control_df.collect()

    succeeded_df = (spark.table(log_table)
                    .filter(col("status") == "SUCCESS")
                    .select("file_name", "tgt_catalog", "tgt_schema", "tgt_table")
                    .distinct())

    pending_df = (control_df.join(
                      succeeded_df,
                      (control_df.Source_File_Name == succeeded_df.file_name) &
                      (control_df.Target_Catalog == succeeded_df.tgt_catalog) &
                      (control_df.Target_Schema == succeeded_df.tgt_schema) &
                      (control_df.Target_Table_Name == succeeded_df.tgt_table),
                      "left_anti"))

    return pending_df.collect()


## Read Source Data
This function reads the source CSV file and appends the `loaded_at` timestamp.


In [0]:
def read_source_file(source_path, source_file_delimiter, file_s_no, date_format,target_table):

    df_schema = spark.read.table(f"{target_table}")
    existing_schema = df_schema.schema
    
    print("-----------target_table_df----------")
    df_schema.printSchema()
    print("-----------existing_schema----------")
    print(existing_schema)

    reader = (
        spark.read
        .option("header", True)
        .schema(existing_schema)
        .option(f"timestampFormat",f"{date_format}")
    )

    if source_file_delimiter == "|":
        reader = (
            reader
            .option("sep", "|")
            .option("quote", '"')
        )

    df = reader.csv(source_path)

    print("-----------df.printSchema()----------")
    df.printSchema()
    return df

## Load Data into Delta Table

This function performs the following:

- Checks whether the target table exists
- Creates a Delta table if it does not exist
- Appends data when no primary key is configured
- Performs Delta MERGE when primary keys are available
- Updates existing records only when the source date is newer

In [0]:
def load_to_delta(df_source, full_tgt, pk_cols, date_col):

    if not pk_cols:

        (
            df_source.write
            .format("delta")
            .mode("append")
            .saveAsTable(full_tgt)
        )

        print(f"Appended into : {full_tgt}")
        return

    if isinstance(pk_cols, str):
        pk_cols = [c.strip() for c in pk_cols.split(",") if c.strip()]

    merge_condition = " AND ".join(
        f"tgt.{c} = src.{c}" for c in pk_cols
        )

    delta_table = DeltaTable.forName(spark, full_tgt)

    merge = (
        delta_table.alias("tgt")
        .merge(df_source.alias("src"), merge_condition)
    )

    if date_col and date_col in df_source.columns:
        merge = merge.whenMatchedUpdateAll(
            condition=f"src.{date_col} > tgt.{date_col}"
        )
    else:
        merge = merge.whenMatchedUpdateAll()

    (
        merge
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged into : {full_tgt}")

# Validate Row Count

Compares the expected row count from the control table with the actual row count from the source file.

Returns an error message if the counts do not match.

In [0]:
def validate_row_count(expected, actual):

    if expected is None:
        return None

    if expected != actual:
        return (
            f"Row Count Mismatch. "
            f"Expected={expected}, Actual={actual}"
        )

    return None

## Log
- Uses an explicit schema (`log_schema`) rather than type inference
- Appends log rows immediately (failure won't break the process).

In [0]:
def write_log(record: dict):

    ordered = {field.name: record.get(field.name) for field in log_schema.fields}
    log_df = spark.createDataFrame([Row(**ordered)], schema=log_schema)
    log_df.write.format("delta").mode("append").saveAsTable(log_table)

- `base_log_record()` - maps control-table columns to the log-table's internal fields.__


In [0]:
def base_log_record(r, batch_id, full_source_path):
    return {
        "log_id": str(uuid.uuid4()),
        "batch_id": batch_id,
        "s_no": r["S_No"],
        "datamart": r["DataMart"],
        "file_name": r["Source_File_Name"],
        "source_path": full_source_path,
        "tgt_catalog": r["Target_Catalog"],
        "tgt_schema": r["Target_Schema"],
        "tgt_table": r["Target_Table_Name"],
        "expected_count": int(r["File_Count"]),
    }

## Log Successful Execution

Writes a SUCCESS record into the ingestion log table.

Logged details include:

- Batch ID
- File Name
- Start Time
- End Time
- Duration
- Row Count
- Status
- Executed By

In [0]:
def log_success(log_record, row_count, error_message,
                executed_by, start_time):

    end_time = datetime.now()

    log_record.update({
        "row_count": row_count,
        "status": "SUCCESS",
        "error_message": error_message,
        "executed_by": executed_by,
        "start_time": start_time,
        "end_time": end_time,
        "loaded_at": end_time,
        "duration_sec": (end_time - start_time).total_seconds()
    })

    write_log(log_record)

## Log Failed Execution

Captures exception details and writes a FAILED record into the ingestion log.

Logged details include:

- Error Message
- Execution Duration
- Start Time
- End Time
- Status

In [0]:
def log_failure(log_record, executed_by, start_time):

    end_time = datetime.now()

    log_record.update({
        "row_count": None,
        "status": "FAILED",
        "error_message": traceback.format_exc()[:4000],
        "executed_by": executed_by,
        "start_time": start_time,
        "end_time": end_time,
        "loaded_at": end_time,
        "duration_sec": (end_time - start_time).total_seconds()
    })

    write_log(log_record)